# NoClip · Difix 실험 (Google Colab)

방 하나를 **① 기본 학습 → ② Difix 보정 학습**으로 돌려서, 벽 쪽(옆) 각도의 줄무늬·비침이 줄어드는지 비교함.
③ 배경 무작위화(해결 방향 3)도 같은 환경에서 선택적으로 돌릴 수 있음.

**준비 (PC에서 한 번)**
1. `F:\Sample\6\6b\data` 폴더 안의 `images`, `sparse` 를
   `G:\내 드라이브\3DGS\colab\room6` 에 복사 (구글 드라이브 동기화로 자동 업로드)
2. Colab 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**

**순서**: 위에서부터 셀 실행. `2. 설치` 후에는 **세션 다시 시작**이 한 번 필요함 (안내가 뜸).
결과(체크포인트, PLY, 비교 이미지)는 전부 드라이브 `3DGS/colab/out` 에 저장되므로 중간에 끊겨도 남아 있음.

## 1. 드라이브 연결 · 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SCENE      = "room6"                                   # 드라이브 3DGS/colab/ 아래 폴더 이름
DRIVE_DATA = f"/content/drive/MyDrive/3DGS/colab/{SCENE}"
OUT        = f"/content/drive/MyDrive/3DGS/colab/out/{SCENE}"
DATA       = f"/content/data/{SCENE}"                  # 빠른 로컬 복사본
FACTOR     = 4      # 4K 원본 기준 4 -> 960x540 (T4 권장). 최종 화질은 2 (1920x1080)
SH         = 2      # Brush 와 동일

import os
os.environ.update(SCENE=SCENE, DRIVE_DATA=DRIVE_DATA, OUT=OUT, DATA=DATA, FACTOR=str(FACTOR), SH=str(SH))
os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls "$DRIVE_DATA"

## 2. 설치 (10~20분, 처음 한 번)
끝나면 **런타임 → 세션 다시 시작** 후 1번 셀부터 다시 실행 (설치 셀은 건너뜀).

In [ ]:
%cd /content
!git clone -q https://github.com/nv-tlabs/Difix3D.git
!pip install -q ninja
!pip install -q diffusers==0.25.1 transformers==4.38.0 huggingface-hub==0.25.1 peft==0.9.0 \
    einops lpips "imageio[ffmpeg]" viser nerfview==0.0.2 "tyro>=0.8.8" tensorly scikit-learn \
    "torchmetrics[image]" tensorboard opencv-python pyyaml
!pip install -q git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e
!pip install -q --no-build-isolation git+https://github.com/rahul-goel/fused-ssim@1272e21a282342e89537159e4bad508b19b34157
!pip install -q gsplat
print("설치 끝 -> 런타임 > 세션 다시 시작")

## 3. 데이터 준비 (드라이브 → 로컬, 축소 이미지 생성)

In [ ]:
import os, shutil, glob
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
if not os.path.exists(f"{DATA}/sparse/0"):
    shutil.copytree(f"{DRIVE_DATA}/sparse", f"{DATA}/sparse")
    shutil.copytree(f"{DRIVE_DATA}/images", f"{DATA}/images")
src = sorted(glob.glob(f"{DATA}/images/*"))
dst_dir = f"{DATA}/images_{FACTOR}"
os.makedirs(dst_dir, exist_ok=True)
def shrink(p):
    q = os.path.join(dst_dir, os.path.basename(p))
    if os.path.exists(q): return
    im = Image.open(p); w, h = im.size
    im.resize((w // FACTOR, h // FACTOR), Image.LANCZOS).save(q, quality=95)
with ThreadPoolExecutor(8) as ex: list(ex.map(shrink, src))
print(len(src), "장 ->", dst_dir, Image.open(glob.glob(dst_dir + "/*")[0]).size)

## 4. 학습 코드 수정 · 보조 스크립트 만들기
보정할 각도 = 학습 카메라를 벽 쪽(좌우)으로 돌린 각도. 아래 셀 5개를 순서대로 실행.

In [ ]:
import os; os.makedirs('/content/nc', exist_ok=True)

In [ ]:
%%writefile /content/nc/nc_patch.py
"""Difix3D gsplat 트레이너를 우리 촬영 데이터(복도/방)에 맞게 수정.
1) Difix 모델은 처음 보정할 때만 불러옴 (기본 학습 중 GPU 메모리 절약)
2) 보정할 '새 각도' = 학습 카메라를 좌우로 돌려 벽을 보게 한 각도.
   라운드마다 NC_STEP_DEG 씩 더 돌리고, 최대 NC_MAX_DEG 까지.
"""
import sys

path = sys.argv[1]
src = open(path, encoding="utf-8").read()
if "[NoClip]" in src:
    print("이미 수정됨"); sys.exit(0)

old_init = '''        # Diffusion fixer
        self.difix = DifixPipeline.from_pretrained("nvidia/difix_ref", trust_remote_code=True)
        self.difix.set_progress_bar_config(disable=True)
        self.difix.to("cuda")
'''
new_init = '''        # Diffusion fixer (loaded lazily on first fix)
        self.difix = None
'''
old_fix = '''        if len(self.cfg.fix_steps) == 1:
            novel_poses = self.parser.camtoworlds[self.valset.indices]
        else:
            novel_poses = self.interpolator.shift_poses(self.current_novel_poses, self.parser.camtoworlds[self.valset.indices], distance=0.5)
'''
new_fix = '''        if self.difix is None:
            self.difix = DifixPipeline.from_pretrained("nvidia/difix_ref", trust_remote_code=True)
            self.difix.set_progress_bar_config(disable=True)
            self.difix.to("cuda")
        # [NoClip] look sideways toward the walls, a little more every round
        k = len(self.novelloaders) + 1
        max_deg = float(os.environ.get("NC_MAX_DEG", 60))
        step_deg = float(os.environ.get("NC_STEP_DEG", 15))
        stride = int(os.environ.get("NC_STRIDE", 8))
        ang = np.deg2rad(min(max_deg, k * step_deg))
        base = self.parser.camtoworlds[np.array(self.trainset.indices)[::stride]]
        rotated = []
        for sgn in (1.0, -1.0):
            c, s = np.cos(sgn * ang), np.sin(sgn * ang)
            Ry = np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])
            p = base.copy()
            p[:, :3, :3] = base[:, :3, :3] @ Ry  # yaw around the camera's own up/down axis
            rotated.append(p)
        novel_poses = np.concatenate(rotated, 0)
        print(f"[NoClip] fix round {k}: {len(novel_poses)} views, yaw +-{np.rad2deg(ang):.0f} deg")
'''
old_alpha = '''        parser.alpha_mask_paths = [f"{self.render_dir}/novel/{step}/Alpha/{i:04d}.png" for i in range(len(novel_poses))]
'''
new_alpha = '''        parser.alpha_mask_paths = [f"{self.render_dir}/novel/{step}/Alpha/{i:04d}.png" for i in range(len(novel_poses))]
        if os.environ.get("NC_NO_ALPHA") == "1":  # [NoClip] also learn where the render was empty
            parser.alpha_mask_paths = None
'''
for o, n in [(old_init, new_init), (old_fix, new_fix), (old_alpha, new_alpha)]:
    assert src.count(o) == 1, "패치 위치를 찾지 못함: " + o[:60]
    src = src.replace(o, n)
open(path, "w", encoding="utf-8").write(src)
print("수정 완료:", path)

In [ ]:
%%writefile /content/nc/nc_export_ply.py
"""gsplat 체크포인트(.pt) -> 표준 3DGS PLY (Unity / SuperSplat 용)
사용: python nc_export_ply.py ckpt_59999_rank0.pt out.ply
"""
import sys
import numpy as np
import torch

ckpt, out = sys.argv[1], sys.argv[2]
s = torch.load(ckpt, map_location="cpu", weights_only=True)["splats"]
means = s["means"].float().numpy()
scales = s["scales"].float().numpy()                       # log scale
quats = torch.nn.functional.normalize(s["quats"].float(), dim=-1).numpy()  # w x y z
opac = s["opacities"].float().numpy()[:, None]             # logit
sh0 = s["sh0"].float().numpy().reshape(len(means), 3)
shN = s["shN"].float().numpy()                              # [N, K, 3]
f_rest = shN.transpose(0, 2, 1).reshape(len(means), -1)    # channel-major like original 3DGS
normals = np.zeros_like(means)

names = ["x", "y", "z", "nx", "ny", "nz"]
names += [f"f_dc_{i}" for i in range(3)]
names += [f"f_rest_{i}" for i in range(f_rest.shape[1])]
names += ["opacity"] + [f"scale_{i}" for i in range(3)] + [f"rot_{i}" for i in range(4)]
data = np.concatenate([means, normals, sh0, f_rest, opac, scales, quats], 1).astype(np.float32)
keep = np.isfinite(data).all(1)
data = data[keep]

with open(out, "wb") as f:
    head = "ply\nformat binary_little_endian 1.0\n"
    head += f"element vertex {len(data)}\n"
    head += "".join(f"property float {n}\n" for n in names)
    head += "end_header\n"
    f.write(head.encode("ascii"))
    f.write(data.tobytes())
print(f"{out}: {len(data):,} splats (SH degree {int(np.sqrt(shN.shape[1] + 1)) - 1})")

In [ ]:
%%writefile /content/nc/nc_compare.py
"""같은 위치에서 카메라를 옆으로 돌려(벽 쪽) 여러 결과를 나란히 렌더링 -> PNG
사용 (examples/gsplat 폴더에서):
  python nc_compare.py DATA_DIR FACTOR OUT.png 이름1=ckpt1.pt 이름2=ckpt2.pt ...
"""
import sys
import numpy as np
import torch
from PIL import Image, ImageDraw
from gsplat.rendering import rasterization
from datasets.colmap import Parser

data_dir, factor, out = sys.argv[1], int(sys.argv[2]), sys.argv[3]
runs = [a.split("=", 1) for a in sys.argv[4:]]
YAWS = [0, 45, 70]                 # 정면(학습 각도), 45도, 70도
N_CAMS = 4

p = Parser(data_dir=data_dir, factor=factor, normalize=False, test_every=0)
K = torch.from_numpy(list(p.Ks_dict.values())[0]).float().cuda()
W, H = list(p.imsize_dict.values())[0]
idx = np.linspace(0, len(p.camtoworlds) - 1, N_CAMS + 2).astype(int)[1:-1]

def yaw(c2w, deg):
    a = np.deg2rad(deg); c, s = np.cos(a), np.sin(a)
    r = c2w.copy(); r[:3, :3] = c2w[:3, :3] @ np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return r

views = [yaw(p.camtoworlds[i], d) for i in idx for d in YAWS]
cols = []
for name, path in runs:
    s = torch.load(path, map_location="cuda", weights_only=True)["splats"]
    colors = torch.cat([s["sh0"], s["shN"]], 1)
    shd = int(np.sqrt(colors.shape[1])) - 1
    tiles = []
    for v in views:
        vm = torch.linalg.inv(torch.from_numpy(v).float().cuda())[None]
        with torch.no_grad():
            img, alpha, _ = rasterization(s["means"], s["quats"], torch.exp(s["scales"]),
                                          torch.sigmoid(s["opacities"]), colors, vm, K[None],
                                          W, H, sh_degree=shd)
        # 비어 있는 곳을 분홍으로 (Unity 분홍 배경 테스트와 같은 의미)
        pink = torch.tensor([1.0, 0.0, 1.0], device="cuda")
        rgb = img[0].clamp(0, 1) + pink * (1 - alpha[0])
        tiles.append(Image.fromarray((rgb.clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)))
    cols.append((name, tiles))
    del s; torch.cuda.empty_cache()

tw, th = W // 2, H // 2
canvas = Image.new("RGB", (tw * len(cols), (th) * len(views) + 40), "white")
d = ImageDraw.Draw(canvas)
for c, (name, tiles) in enumerate(cols):
    d.text((c * tw + 10, 10), name, fill="black")
    for r, t in enumerate(tiles):
        canvas.paste(t.resize((tw, th)), (c * tw, 40 + r * th))
canvas.save(out)
print("saved", out, "| rows = 카메라 x [정면, 45도, 70도], 분홍 = 비어 있는 곳")

In [ ]:
!python /content/nc/nc_patch.py /content/Difix3D/examples/gsplat/simple_trainer_difix3d.py
!cp /content/nc/nc_compare.py /content/Difix3D/examples/gsplat/
# Colab 에 깔린 HuggingFace 'datasets' 패키지와 이름이 겹쳐서, 로컬 폴더를 정식 패키지로 만들어 우선시킴
open("/content/Difix3D/examples/gsplat/datasets/__init__.py", "w").close()
# pycolmap 을 numpy 2 에서 쓸 수 있게 한 줄 수정 (np.uint64(-1) -> 같은 값)
import glob, site
for f in glob.glob("/usr/local/lib/python3*/dist-packages/pycolmap/scene_manager.py"):
    s = open(f).read().replace("np.uint64(-1)", "np.uint64(np.iinfo(np.uint64).max)")
    open(f, "w").write(s); print("patched", f)

## 5. ① 기본 학습 (보정 없음, 30k 스텝 · T4 약 1시간)
비교 기준. Brush 결과와 같은 역할.

처음 실행 때 gsplat CUDA 코드 컴파일로 10분 정도 멈춘 것처럼 보일 수 있음 (정상).

In [ ]:
%cd /content/Difix3D/examples/gsplat
!PYTHONPATH=/content/Difix3D python simple_trainer_difix3d.py default \
    --data_dir $DATA --data_factor $FACTOR --result_dir $OUT/base \
    --no-normalize-world-space --test_every 0 --sh_degree $SH \
    --max_steps 30000 --fix_steps 1000000 \
    --eval_steps 30000 --save_steps 10000 20000 30000

## 6. ② Difix 보정 학습 (①에 이어서 30k 스텝 · T4 약 1.5~2시간)
2,000 스텝마다: 카메라를 좌우로 돌린 각도에서 렌더링 → Difix 로 줄무늬 제거 → 학습 데이터에 추가.
각도는 15° → 30° → 45° → 60° 순으로 점점 벌어짐 (`NC_STEP_DEG`, `NC_MAX_DEG`).

In [ ]:
%cd /content/Difix3D/examples/gsplat
!NC_STEP_DEG=15 NC_MAX_DEG=60 NC_STRIDE=8 PYTHONPATH=/content/Difix3D python simple_trainer_difix3d.py default \
    --data_dir $DATA --data_factor $FACTOR --result_dir $OUT/difix \
    --no-normalize-world-space --test_every 0 --sh_degree $SH \
    --max_steps 60000 --eval_steps 60000 --save_steps 40000 50000 60000 \
    --ckpt $OUT/base/ckpts/ckpt_29999_rank0.pt

## 7. ③ (선택) 배경 무작위화 + 스케일 정규화 — 해결 방향 3
Brush 에 없던 배경 무작위화를 여기서 테스트. ①과 같은 조건에 옵션만 추가.

In [ ]:
%cd /content/Difix3D/examples/gsplat
!PYTHONPATH=/content/Difix3D python simple_trainer_difix3d.py default \
    --data_dir $DATA --data_factor $FACTOR --result_dir $OUT/rbg \
    --no-normalize-world-space --test_every 0 --sh_degree $SH \
    --max_steps 30000 --fix_steps 1000000 \
    --eval_steps 30000 --save_steps 10000 20000 30000 \
    --random_bkgd --scale_reg 0.01

## 8. 비교 이미지 (같은 위치에서 정면 · 45° · 70°, 분홍 = 비어 있는 곳)

In [ ]:
%cd /content/Difix3D/examples/gsplat
import os
runs = {"base": f"{OUT}/base/ckpts/ckpt_29999_rank0.pt",
        "difix": f"{OUT}/difix/ckpts/ckpt_59999_rank0.pt",
        "rbg": f"{OUT}/rbg/ckpts/ckpt_29999_rank0.pt"}
args = " ".join(f"{k}={v}" for k, v in runs.items() if os.path.exists(v))
!python nc_compare.py $DATA $FACTOR $OUT/compare.png {args}
from IPython.display import Image as Show
Show(f"{OUT}/compare.png")

## 9. PLY 내보내기 → Unity / SuperSplat
드라이브 `3DGS/colab/out/<방>/ply` 에 저장됨. PC에서 그대로 Unity 로 가져오면 됨.

In [ ]:
import os
runs = {"base": f"{OUT}/base/ckpts/ckpt_29999_rank0.pt",
        "difix": f"{OUT}/difix/ckpts/ckpt_59999_rank0.pt",
        "rbg": f"{OUT}/rbg/ckpts/ckpt_29999_rank0.pt"}
os.makedirs(f"{OUT}/ply", exist_ok=True)
for k, v in runs.items():
    if os.path.exists(v):
        !python /content/nc/nc_export_ply.py {v} {OUT}/ply/{SCENE}_{k}.ply